# Depth-to-Elevation Calibration

This notebook uses Depth Anything V2 to produce a monocular depth map from a camera image, then calibrates it against known water surface elevations to create a full-scene elevation map in real-world units (ft).

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
from PIL import Image
import cv2
import torch

from depth_anything_v2.dpt import DepthAnythingV2

## Existing Runs

Runs with an existing `run_config.json`:

In [ ]:
for cfg in sorted(glob.glob('cameras/*/*/run_config.json')):
    parts = cfg.split(os.sep)
    print(f"camera_id = '{parts[1]}'")
    print(f"run_name  = '{parts[2]}'")
    print()

## Configuration

In [ ]:
# Camera and directory configuration
camera_id = 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet'
run_name  = 'event_2025-06-23'
run_dir = f'cameras/{camera_id}/{run_name}'

images_dir = f'{run_dir}/images'
masks_dir = f'{run_dir}/masks'
csv_path = f'{run_dir}/images_and_data.csv'

# Depth Anything V2 configuration
encoder = 'vits'  # 'vits' (Small), 'vitb' (Base), or 'vitl' (Large)
checkpoint_path = f'../checkpoints/depth_anything_v2_{encoder}.pth'

# Output directory
output_dir = f'{run_dir}/depth'
os.makedirs(output_dir, exist_ok=True)

print(f"Loading data from: {csv_path}")
print(f"Output will be saved to: {output_dir}")

## Load Data

In [ ]:
# Load the CSV with images, masks, and gage height data
df = pd.read_csv(csv_path)

print(f"Loaded {len(df)} records")

# Filter out bad quality masks if quality_flag column exists
if 'quality_flag' in df.columns:
    n_bad = (df['quality_flag'] == 'bad').sum()
    if n_bad > 0:
        print(f"\nFound {n_bad} masks marked as 'bad' - excluding from analysis")
        bad_indices = df[df['quality_flag'] == 'bad'].index.tolist()
        print(f"Excluded indices: {bad_indices}")
        df = df[df['quality_flag'] != 'bad'].reset_index(drop=True)
        print(f"Remaining records after filtering: {len(df)}")
    else:
        print(f"\nAll masks marked as 'good' - no filtering applied")
else:
    print(f"\nNo quality_flag column found - using all masks")
    print(f"(Run notebook 02.5 to review and flag bad masks)")

print(f"\nGage height (00065) range: {df['00065'].min():.2f} to {df['00065'].max():.2f} ft")
print(f"\nFirst few rows:")
display(df[['image_names', 'mask_filename', '00065']].head())

## Depth Estimation

Run Depth Anything V2 on a single reference image to produce a relative depth map. Since the camera is fixed, one inference covers the entire scene geometry.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}

model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(checkpoint_path, map_location='cpu'))
model = model.to(DEVICE).eval()
print(f"Loaded Depth Anything V2 ({encoder}) model")

In [ ]:
# Run inference on the first image
reference_image_name = df.iloc[0]['image_names']
reference_image_path = os.path.join(images_dir, reference_image_name)
raw_img = cv2.imread(reference_image_path)

print(f"Reference image: {reference_image_name}")
print(f"Image size: {raw_img.shape[1]} x {raw_img.shape[0]}")

depth = model.infer_image(raw_img)  # HxW relative depth map

# Save relative depth map
np.save(os.path.join(output_dir, 'relative_depth.npy'), depth)
print(f"Depth map shape: {depth.shape}")
print(f"Depth range: {depth.min():.2f} to {depth.max():.2f}")

In [ ]:
# Display reference image and depth map side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Reference Image')
axes[0].axis('off')

im = axes[1].imshow(depth, cmap='viridis')
axes[1].set_title('Relative Depth Map')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label='Relative Depth')

plt.tight_layout()
plt.show()

## Extract Calibration Data

For each water mask, extract the boundary contour pixels and sample the relative depth map at those locations. This pairs known gage heights with depth values to build a calibration dataset.

In [ ]:
calibration_points = []

for idx, row in df.iterrows():
    mask_path = os.path.join(masks_dir, row['mask_filename'])
    if not os.path.exists(mask_path):
        print(f"Warning: Mask not found: {mask_path}")
        continue

    mask = np.load(mask_path)
    if mask.ndim == 3:
        mask = mask.squeeze()
    mask_uint8 = (mask.astype(bool) * 255).astype(np.uint8)

    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    elevation = row['00065']

    for contour in contours:
        pts = contour.squeeze()
        if pts.ndim != 2:
            continue
        xs, ys = pts[:, 0], pts[:, 1]
        # Clip to depth map bounds
        valid = (ys >= 0) & (ys < depth.shape[0]) & (xs >= 0) & (xs < depth.shape[1])
        xs, ys = xs[valid], ys[valid]
        depths = depth[ys, xs]
        for x, y, d in zip(xs, ys, depths):
            calibration_points.append((x, y, d, elevation))

cal_df = pd.DataFrame(calibration_points, columns=['pixel_x', 'pixel_y', 'relative_depth', 'elevation'])
cal_df.to_csv(os.path.join(output_dir, 'calibration_data.csv'), index=False)

print(f"Calibration points: {len(cal_df)}")
print(f"Depth range: {cal_df['relative_depth'].min():.2f} to {cal_df['relative_depth'].max():.2f}")
print(f"Elevation range: {cal_df['elevation'].min():.2f} to {cal_df['elevation'].max():.2f} ft")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(cal_df['relative_depth'], cal_df['elevation'], s=1, alpha=0.1)
ax.set_xlabel('Relative Depth')
ax.set_ylabel('Gage Height (ft)')
ax.set_title('Relative Depth vs. Known Elevation (Calibration Data)')
plt.tight_layout()
plt.show()

## Fit Calibration

Fit a polynomial regression from relative depth to gage height (ft). Degree 2 captures the nonlinear perspective relationship while remaining robust with limited data.

In [ ]:
poly_degree = 2
coeffs = np.polyfit(cal_df['relative_depth'], cal_df['elevation'], deg=poly_degree)
poly_fn = np.poly1d(coeffs)

# Quality metrics
predicted = poly_fn(cal_df['relative_depth'])
residuals = cal_df['elevation'] - predicted
ss_res = np.sum(residuals ** 2)
ss_tot = np.sum((cal_df['elevation'] - cal_df['elevation'].mean()) ** 2)
r_squared = 1 - ss_res / ss_tot
mae = np.mean(np.abs(residuals))
max_err = np.max(np.abs(residuals))

print(f"Polynomial degree: {poly_degree}")
print(f"Coefficients: {coeffs}")
print(f"R²: {r_squared:.4f}")
print(f"MAE: {mae:.3f} ft")
print(f"Max error: {max_err:.3f} ft")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(cal_df['relative_depth'], cal_df['elevation'], s=1, alpha=0.1, label='Calibration data')

d_range = np.linspace(cal_df['relative_depth'].min(), cal_df['relative_depth'].max(), 200)
ax.plot(d_range, poly_fn(d_range), 'r-', linewidth=2, label=f'Poly fit (deg={poly_degree}, R²={r_squared:.3f})')

ax.set_xlabel('Relative Depth')
ax.set_ylabel('Gage Height (ft)')
ax.set_title('Calibration Fit: Relative Depth → Elevation')
ax.legend()
plt.tight_layout()
plt.show()

## Create Elevation Map

Apply the fitted polynomial to the full depth map to produce a calibrated elevation raster.

**Note:** Elevation values outside the observed gage height range are extrapolated and may be unreliable.

In [ ]:
elevation_map = poly_fn(depth)
np.save(os.path.join(output_dir, 'calibrated_elevation.npy'), elevation_map)

print(f"Elevation map shape: {elevation_map.shape}")
print(f"Elevation range: {elevation_map.min():.2f} to {elevation_map.max():.2f} ft")
print(f"Observed gage height range: {cal_df['elevation'].min():.2f} to {cal_df['elevation'].max():.2f} ft")

In [ ]:
# Elevation map overlaid on reference image
reference_rgb = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
min_elev, max_elev = cal_df['elevation'].min(), cal_df['elevation'].max()
norm = Normalize(vmin=min_elev, vmax=max_elev)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(reference_rgb)
im = ax.imshow(elevation_map, cmap='viridis', alpha=0.5, norm=norm)
ax.set_title(f'Calibrated Elevation Map\n{camera_id}\nGage Height Range: {min_elev:.2f} - {max_elev:.2f} ft')
ax.axis('off')
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Gage Height (ft)')

output_path = os.path.join(output_dir, 'elevation_map_with_background.png')
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {output_path}")

In [ ]:
# Clean elevation map (no background)
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(elevation_map, cmap='viridis', norm=norm)
ax.set_title(f'Calibrated Elevation Map (Clean)\n{camera_id}')
ax.axis('off')
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Gage Height (ft)')

output_path = os.path.join(output_dir, 'elevation_map_clean.png')
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {output_path}")

## Summary

In [ ]:
print(f"Output files saved to: {output_dir}")
for f in sorted(os.listdir(output_dir)):
    print(f"  - {f}")